In [78]:
import pandas as pd
import numpy as np

# --- Load ---
df_uhcd = pd.read_csv("Datasets/nda_uhcd_2022_v1.csv", dtype={'nda': str})
df_full = pd.read_csv("Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv", dtype={'nda': str})

# --- Clean NDA ---
df_uhcd['nda'] = df_uhcd['nda'].astype(str).str.strip()
df_full['nda'] = df_full['nda'].astype(str).str.strip()

# --- Filter 2022 only ---
df_full['datetime_admission'] = pd.to_datetime(df_full['datetime_admission'], errors='coerce')
df_2022 = df_full[df_full['datetime_admission'].dt.year == 2022].copy()
print(f"Patients 2022 : {len(df_2022):,}")

# --- Merge ---
df_merged = pd.merge(df_2022, df_uhcd, on='nda', how='left')
print(f"Après merge : {len(df_merged):,}")

# --- Comparaison ---
print("\n=== Colonnes disponibles pour comparaison ===")
for col in ['decision_urgence', 'disposition_med', 'UHCD']:
    present = col in df_merged.columns
    print(f"  {col:20} : {'✅' if present else '❌ manquante'}")

/tmp/ipykernel_4157696/1730446250.py:6: DtypeWarning: Columns (12,78,81,84,85,87,88,89,90,91,92,96,98,100,101,103,105,106,107,108,110,111,112,113,114,115,116,117,118,119,120,121,122,125,131,132,133,136,137,140,143,144,145) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv("Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv", dtype={'nda': str})


Patients 2022 : 31,087
Après merge : 31,087

=== Colonnes disponibles pour comparaison ===
  decision_urgence     : ✅
  disposition_med      : ✅
  UHCD                 : ✅


In [79]:
print(df_full['triage'].value_counts(dropna=False).to_string())
print(df_merged['triage'].value_counts(dropna=False).to_string())

triage
3.0    48593
4.0    33468
2.0    30428
5.0     9698
1.0      984
NaN       18
triage
3.0    11532
4.0     9456
2.0     7766
5.0     2220
1.0      108
NaN        5


In [80]:
# --- Comparaison decision_urgence vs disposition_med vs UHCD ---

print(f"Total patients 2022 mergés : {len(df_merged):,}")

print("\n=== decision_urgence ===")
print(df_merged['decision_urgence'].value_counts(dropna=False).to_string())

print("\n=== disposition_med ===")
print(df_merged['disposition_med'].value_counts(dropna=False).to_string())

print("\n=== UHCD ===")
print(df_merged['UHCD'].value_counts(dropna=False).to_string())

print("\n=== CROISEMENT decision_urgence × UHCD ===")
print(pd.crosstab(
    df_merged['decision_urgence'].fillna('NaN'),
    df_merged['UHCD'].fillna('NaN'),
    margins=True
).to_string())

print("\n=== CROISEMENT disposition_med × UHCD ===")
print(pd.crosstab(
    df_merged['disposition_med'].fillna('NaN'),
    df_merged['UHCD'].fillna('NaN'),
    margins=True
).to_string())

print("\n=== CROISEMENT disposition_med × decision_urgence ===")
print(pd.crosstab(
    df_merged['disposition_med'].fillna('NaN'),
    df_merged['decision_urgence'].fillna('NaN'),
    margins=True
).to_string())

Total patients 2022 mergés : 31,087

=== decision_urgence ===
decision_urgence
Hospitalisation    15566
Consultation       15228
NaN                  292
Pas de décision        1

=== disposition_med ===
disposition_med
Retour domicile                     17291
NaN                                  9167
Hospitalisation au CHU               2421
UHCD puis hospitalisation au CHU      847
UHCD puis RAD                         726
Transfert hors CHU                    474
UHCD puis transfert hors CHU          161

=== UHCD ===
UHCD
False    26458
True      4626
NaN          3

=== CROISEMENT decision_urgence × UHCD ===
UHCD              False  True  NaN    All
decision_urgence                         
Consultation      15151    77    0  15228
Hospitalisation   11047  4516    3  15566
NaN                 259    33    0    292
Pas de décision       1     0    0      1
All               26458  4626    3  31087

=== CROISEMENT disposition_med × UHCD ===
UHCD                              False  

In [81]:
def harmonize_outcome(row):
    dec = str(row['decision_urgence']).strip() if pd.notna(row['decision_urgence']) else 'NaN'
    dis = str(row['disposition_med']).strip() if pd.notna(row['disposition_med']) else 'NaN'
    uhcd = bool(row['UHCD']) if pd.notna(row['UHCD']) else False

    # NaN total
    if dec == 'NaN' and dis == 'NaN':
        return np.nan

    # --- CONSULTATION ---
    if dec == 'Consultation':
        if dis in [
            'Hospitalisation au CHU',
            'UHCD puis hospitalisation au CHU',
            'NaN',
            'Retour domicile',
            'Transfert hors CHU',
            'UHCD puis RAD',
            'UHCD puis transfert hors CHU']:
            return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis in ['Transfert hors CHU']:
        #     return 'UHCD puis transfert hors CHU' if uhcd else 'Transfert hors CHU'
        # if dis in ['Hospitalisation au CHU', 'NaN']:
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis == 'Retour domicile':
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis == 'UHCD puis RAD':
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis == 'UHCD puis hospitalisation au CHU':
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'
        # if dis in ['UHCD puis transfert hors CHU']:
        #     return 'UHCD puis RAD' if uhcd else 'Retour domicile'

    # --- HOSPITALISATION ---
    if dec == 'Hospitalisation':
        # if dis == 'Transfert hors CHU':
        #     return 'UHCD puis transfert hors CHU' if uhcd else 'Transfert hors CHU'
        if dis in [
            'Hospitalisation au CHU',
            'UHCD puis hospitalisation au CHU',
            'NaN',
            'Retour domicile',
            'Transfert hors CHU',
            'UHCD puis RAD',
            'UHCD puis transfert hors CHU']:
            return 'UHCD puis hospitalisation/transfert' if uhcd else 'Hospitalisation/transfert'
        #if dis == 'UHCD puis RAD':
        #    return 'UHCD puis hospitalisation/transfert' if uhcd else 'hospitalisation/transfert'
        #if dis == 'UHCD puis hospitalisation au CHU':
        #    return 'UHCD puis hospitalisation/transfert' if uhcd else 'Hospitalisation/transfert'
        #if dis == 'UHCD puis transfert hors CHU':
        #    return 'UHCD puis transfert hors CHU' if uhcd else 'Transfert hors CHU'

    return np.nan

df_merged['disposition'] = df_merged.apply(harmonize_outcome, axis=1)

print("=== OUTCOME DISTRIBUTION ===")
print(df_merged['disposition'].value_counts(dropna=False).to_string())

print("\n=== VERIFICATION : disposition × obs ===")
print(pd.crosstab(
    df_merged['disposition'].fillna('NaN'),
    df_merged['UHCD'].fillna('NaN'),
    margins=True
).to_string())

=== OUTCOME DISTRIBUTION ===
disposition
Retour domicile                        15151
Hospitalisation/transfert              11050
UHCD puis hospitalisation/transfert     4516
NaN                                      293
UHCD puis RAD                             77

=== VERIFICATION : disposition × obs ===
UHCD                                 False  True  NaN    All
disposition                                                 
Hospitalisation/transfert            11047     0    3  11050
NaN                                    260    33    0    293
Retour domicile                      15151     0    0  15151
UHCD puis RAD                            0    77    0     77
UHCD puis hospitalisation/transfert      0  4516    0   4516
All                                  26458  4626    3  31087


In [82]:
# def harmonize_disposition(row):
#     dec  = str(row['decision_urgence']).strip() if pd.notna(row['decision_urgence']) else 'NaN'
#     uhcd = bool(row['UHCD']) if pd.notna(row['UHCD']) else False
#
#     if dec == 'Hospitalisation':
#         return 'UHCD puis Hospitalisation' if uhcd else 'Hospitalisation'
#     if dec == 'Consultation':
#         return 'UHCD puis RAD' if uhcd else 'Retour domicile'
#     if dec == 'Hospitalisation':
#         return 'UHCD puis transfert hors CHU' if uhcd else ''
#
#     return np.nan
#
# df_merged['disposition'] = df_merged.apply(harmonize_disposition, axis=1)
#
# print("=== disposition DISTRIBUTION ===")
# print(df_merged['disposition'].value_counts(dropna=False).to_string())

voir si les transferts je les considere comme des hospit ou pas

In [83]:
def fill_nan_disposition(row):
    if pd.notna(row['disposition']):
        return row['disposition']

    dis = str(row['disposition_med']).strip() if pd.notna(row['disposition_med']) else 'NaN'
    uhcd = bool(row['UHCD']) if pd.notna(row['UHCD']) else False


    if dis in [
        'Hospitalisation au CHU',
        'Transfert hors CHU',
        'UHCD puis hospitalisation au CHU',
        'UHCD puis transfert hors CHU']:
        return 'UHCD puis hospitalisation/transfert' if uhcd else 'Hospitalisation/transfert'

    if dis in ['Retour domicile', 'UHCD puis RAD']:
        return 'UHCD puis RAD' if uhcd else 'Retour domicile'



    return np.nan

df_merged['disposition'] = df_merged.apply(fill_nan_disposition, axis=1)

print("=== disposition DISTRIBUTION ===")
print(df_merged['disposition'].value_counts(dropna=False).to_string())

=== disposition DISTRIBUTION ===
disposition
Retour domicile                        15294
Hospitalisation/transfert              11076
UHCD puis hospitalisation/transfert     4528
NaN                                      100
UHCD puis RAD                             89


In [84]:
print("=== CROISEMENT disposition × disposition_med ===")
print(pd.crosstab(
    df_merged['disposition'].fillna('NaN'),
    df_merged['disposition_med'].fillna('NaN'),
    margins=True
).to_string())

=== CROISEMENT disposition × disposition_med ===
disposition_med                      Hospitalisation au CHU   NaN  Retour domicile  Transfert hors CHU  UHCD puis RAD  UHCD puis hospitalisation au CHU  UHCD puis transfert hors CHU    All
disposition                                                                                                                                                                                 
Hospitalisation/transfert                              1775  3143             5841                 242             38                                23                            14  11076
NaN                                                       0   100                0                   0              0                                 0                             0    100
Retour domicile                                          64  4631            10506                  76             13                                 2                             2  15294
UHCD p

In [85]:
# Qui sont les 108 patients avec disposition encore NaN ?
df_nan_disp = df_merged[df_merged['disposition'].isna()].copy()

print(f"Patients avec disposition NaN restants : {len(df_nan_disp):,}")

print("\n=== Score de tri ===")
print(df_nan_disp['triage'].value_counts(dropna=False).to_string())

print("\n=== decision_urgence ===")
print(df_nan_disp['decision_urgence'].value_counts(dropna=False).to_string())

print("\n=== disposition_med ===")
print(df_nan_disp['disposition_med'].value_counts(dropna=False).to_string())

print("\n=== UHCD ===")
print(df_nan_disp['UHCD'].value_counts(dropna=False).to_string())

print("\n=== Croisement decision_urgence x UHCD ===")
print(pd.crosstab(
    df_nan_disp['decision_urgence'].fillna('NaN'),
    df_nan_disp['UHCD'].fillna('NaN'),
    margins=True
).to_string())

Patients avec disposition NaN restants : 100

=== Score de tri ===
triage
3.0    38
2.0    27
4.0    25
5.0    10

=== decision_urgence ===
decision_urgence
NaN    100

=== disposition_med ===
disposition_med
NaN    100

=== UHCD ===
UHCD
False    91
True      9

=== Croisement decision_urgence x UHCD ===
UHCD              False  True  All
decision_urgence                  
NaN                  91     9  100
All                  91     9  100


/tmp/ipykernel_4157696/3106846318.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_nan_disp['UHCD'].fillna('NaN'),


In [86]:
# --- Merge dans le gros dataset ---
# --- Garder seulement 2022 --- (déjà le cas mais sécurité)
df_merged = df_merged[df_merged['datetime_admission'].dt.year == 2022].copy()

print(f"Patients 2022 uniquement : {len(df_merged):,}")
print(f"Columns : {df_merged.columns.tolist()}")

# --- Export ---
df_merged.to_csv("df_adm_pv_ioa_med_radio_labo_uhcd_pel22_tabular.csv", index=False)
print("✅ Saved.")

Patients 2022 uniquement : 31,087
Columns : ['nda', 'sex', 'age', 'uam_service', 'hospital', 'datetime_admission', 'date_sortie_urg', 'date_sortie_chu', 'date_sortie_urg_completee', 'mode_sortie_chu', 'decision_urgence', 'transport_grouped', 'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'chief_complaint', 'triage', 'triage_raw', 'sbp', 'dbp', 'mbp', 'bp_status', 'is_bp_measured', 'hr', 'hr_status', 'is_hr_measured', 'temp', 'temp_status', 'is_temp_measured', 'sat', 'sat_status', 'is_sat_measured', 'rr', 'rr_status', 'is_rr_measured', 'o2_flow', 'o2_flow_status', 'is_o2_measured', 'gcs', 'gcs_status', 'is_gcs_measured', 'cap_blood_sugar_mmol_L', 'cap_blood_sugar_status', 'is_cap_blood_sugar_mmol_L_measured', 'pupil_right', 'pupil_left', 'pupils_status', 'anisocoria_status', 'is_pupils_measured', 'urine_dipstick_clean', 'is_urine_dipstick_clean_measured', 'urine_dipstick_clean_status', 'pain', 'pain_status', 'is_pain_measured', 'breathalyzer', 'breathalyzer

In [87]:
print(df_merged['triage'].value_counts(dropna=False).to_string())

triage
3.0    11532
4.0     9456
2.0     7766
5.0     2220
1.0      108
NaN        5


In [88]:
# # --- Extraire disposition et UHCD de df_merged ---
# df_disposition = df_merged[['nda', 'disposition', 'UHCD']].copy()
#
# # --- Merge dans le gros dataset ---
# df_2022_full = df_2022_full.merge(df_disposition, on='nda', how='left')
#
# # --- Vérification ---
# print(f"Total rows : {len(df_2022_full):,}")
# print(f"disposition renseigné : {df_2022_full['disposition'].notna().sum():,}")
# print(f"UHCD renseigné    : {df_2022_full['UHCD'].notna().sum():,}")
#
# print("\n=== disposition DISTRIBUTION ===")
# print(df_2022_full['disposition'].value_counts(dropna=False).to_string())
#
# # --- Export ---
# df_2022_full.to_csv("Datasets/df_adm_pv_ioa_med_radio_labo_tabular.csv", index=False)
# print("\n✅ Saved to: Datasets/df_adm_pv_ioa_med_radio_labo_uhcd_tabular.csv")